# Clustering Visualization

Visualize character clustering results from the tree refinement algorithm.

**Contents:**
1. Document summary — cluster counts and statistics
2. Cluster means gallery — visual overview of all cluster centroids
3. Character browser — explore characters assigned to each cluster
4. Cluster comparison — side-by-side view of similar clusters
5. Document comparison — compare clustering across documents

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────

# Paths
CHARNET_ROOT = r"..\data\corpus-1\charnet"  # folder with document subfolders
IMAGE_ROOT   = r"..\data\corpus-1\imgs"     # for page overlays

# Document to inspect (subfolder name)
DOCUMENT = "BNE_1001_615_T-55281-18"

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────

import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 9,
    "axes.titlesize": 11,
})

In [ ]:
# ── Load clustering results ───────────────────────────────────────────

doc_dir = Path(CHARNET_ROOT) / DOCUMENT
cluster_path = doc_dir / "clusters_all.npz"
italic_path = doc_dir / "italic_labels.npz"

assert cluster_path.exists(), f"Clusters not found: {cluster_path}"

cluster_data = np.load(str(cluster_path), allow_pickle=True)

cluster_labels = cluster_data["cluster_labels"]   # (N,) cluster ID per char
cluster_means  = cluster_data["cluster_means"]    # (K, 40, 32) centroids
cluster_counts = cluster_data["cluster_counts"]   # (K,) chars per cluster
char_labels    = cluster_data["char_labels"]      # (N,) OCR labels

# Optional: italic labels
if italic_path.exists():
    italic_data = np.load(str(italic_path), allow_pickle=True)
    char_italic = italic_data["char_italic"]
    has_italic = True
else:
    char_italic = np.zeros(len(cluster_labels), dtype=bool)
    has_italic = False

n_chars = len(cluster_labels)
n_clusters = len(cluster_means)

print(f"Document: {DOCUMENT}")
print(f"Characters: {n_chars:,}")
print(f"Clusters: {n_clusters}")
print(f"Italic data: {'yes' if has_italic else 'no'}")

---
## 1. Document Summary

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────

# Cluster size distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of cluster sizes
ax = axes[0]
ax.hist(cluster_counts, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
ax.set_xlabel("Characters per cluster")
ax.set_ylabel("Number of clusters")
ax.set_title("Cluster Size Distribution")
ax.axvline(np.median(cluster_counts), color="red", linestyle="--", 
           label=f"Median: {np.median(cluster_counts):.0f}")
ax.legend()

# Cumulative coverage
ax = axes[1]
sorted_counts = np.sort(cluster_counts)[::-1]
cumsum = np.cumsum(sorted_counts) / n_chars * 100
ax.plot(range(1, len(cumsum) + 1), cumsum, "b-", linewidth=2)
ax.set_xlabel("Number of clusters (sorted by size)")
ax.set_ylabel("Cumulative coverage (%)")
ax.set_title("Character Coverage by Top Clusters")
ax.axhline(90, color="red", linestyle="--", alpha=0.5)
# Find how many clusters cover 90%
n_for_90 = np.searchsorted(cumsum, 90) + 1
ax.axvline(n_for_90, color="red", linestyle="--", alpha=0.5,
           label=f"{n_for_90} clusters for 90% coverage")
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f"\nTop 10 clusters cover {cumsum[9]:.1f}% of characters")
print(f"Top 50 clusters cover {cumsum[min(49, len(cumsum)-1)]:.1f}% of characters")

In [ ]:
# ── Per-cluster OCR label breakdown ───────────────────────────────────

def get_cluster_label_dist(cluster_id):
    """Get OCR label distribution for a cluster."""
    mask = cluster_labels == cluster_id
    labels = char_labels[mask]
    unique, counts = np.unique(labels, return_counts=True)
    order = np.argsort(counts)[::-1]
    return list(zip(unique[order], counts[order]))

# Build summary table
rows = []
for cid in range(n_clusters):
    dist = get_cluster_label_dist(cid)
    top_label = dist[0][0] if dist else "?"
    top_pct = 100 * dist[0][1] / cluster_counts[cid] if dist else 0
    
    # Italic percentage if available
    mask = cluster_labels == cid
    italic_pct = 100 * char_italic[mask].mean() if has_italic else 0
    
    rows.append({
        "cluster": cid,
        "count": int(cluster_counts[cid]),
        "top_label": top_label,
        "top_pct": top_pct,
        "italic_pct": italic_pct,
    })

df = pd.DataFrame(rows).sort_values("count", ascending=False)

print("Top 20 clusters by size:")
df.head(20)

---
## 2. Cluster Means Gallery

Visual overview of all cluster centroids, sorted by size.

In [ ]:
# ── Plot cluster means ────────────────────────────────────────────────

# Sort by cluster size
order = np.argsort(cluster_counts)[::-1]

# Determine grid size
n_show = min(100, n_clusters)  # show at most 100
cols = 10
rows_grid = (n_show + cols - 1) // cols

fig, axes = plt.subplots(rows_grid, cols, figsize=(cols * 1.2, rows_grid * 1.5))
axes = np.atleast_2d(axes)

for idx in range(rows_grid * cols):
    row, col = idx // cols, idx % cols
    ax = axes[row, col]
    
    if idx < n_show:
        cid = order[idx]
        ax.imshow(cluster_means[cid], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"#{cid}\n({cluster_counts[cid]})", fontsize=7)
    ax.set_axis_off()

fig.suptitle(f"Cluster Means — {DOCUMENT} ({n_clusters} total)", fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---
## 3. Character Browser

Explore characters assigned to a specific cluster.

In [ ]:
# ── Load character images from page data ──────────────────────────────

def load_all_char_imgs(doc_dir):
    """Load all character images from page .npz files."""
    all_imgs = []
    for npz_path in sorted(doc_dir.glob("*_data.npz")):
        data = np.load(str(npz_path), allow_pickle=True)
        if "char_imgs" in data:
            all_imgs.append(data["char_imgs"])
    return np.concatenate(all_imgs) if all_imgs else np.array([])

char_imgs = load_all_char_imgs(doc_dir)
print(f"Loaded {len(char_imgs)} character images")

# Sanity check
assert len(char_imgs) == n_chars, f"Mismatch: {len(char_imgs)} imgs vs {n_chars} labels"

In [ ]:
# ── Select cluster to browse ──────────────────────────────────────────

CLUSTER_ID = 0  # Change this to browse different clusters
MAX_SHOW = 64   # Maximum characters to display

mask = cluster_labels == CLUSTER_ID
indices = np.where(mask)[0]
n_in_cluster = len(indices)

print(f"Cluster {CLUSTER_ID}: {n_in_cluster} characters")

# Label distribution
dist = get_cluster_label_dist(CLUSTER_ID)
print(f"OCR labels: {dict(dist[:5])}{'...' if len(dist) > 5 else ''}")

if has_italic:
    italic_pct = 100 * char_italic[mask].mean()
    print(f"Italic: {italic_pct:.1f}%")

In [ ]:
# ── Display characters in cluster ─────────────────────────────────────

show_indices = indices[:MAX_SHOW]
n_show = len(show_indices)

cols = 8
rows_grid = (n_show + cols - 1) // cols

fig, axes = plt.subplots(rows_grid, cols, figsize=(cols * 1.0, rows_grid * 1.2))
axes = np.atleast_2d(axes)

for idx in range(rows_grid * cols):
    row, col = idx // cols, idx % cols
    ax = axes[row, col]
    
    if idx < n_show:
        char_idx = show_indices[idx]
        ax.imshow(char_imgs[char_idx], cmap="gray", vmin=0, vmax=1)
        lbl = char_labels[char_idx]
        ax.set_title(lbl, fontsize=8, color="red" if has_italic and char_italic[char_idx] else "black")
    ax.set_axis_off()

fig.suptitle(f"Cluster {CLUSTER_ID} — {n_in_cluster} chars (showing {n_show})", fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# ── Compare cluster mean vs actual samples ────────────────────────────

fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))

# Cluster mean
axes[0].imshow(cluster_means[CLUSTER_ID], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Mean", fontweight="bold")
axes[0].set_axis_off()

# Random samples
sample_idx = np.random.choice(indices, size=min(5, len(indices)), replace=False)
for i, idx in enumerate(sample_idx):
    axes[i + 1].imshow(char_imgs[idx], cmap="gray", vmin=0, vmax=1)
    axes[i + 1].set_title(char_labels[idx])
    axes[i + 1].set_axis_off()

fig.suptitle(f"Cluster {CLUSTER_ID}: Mean vs Samples", fontweight="bold")
fig.tight_layout()
plt.show()

---
## 4. Cluster Similarity

Find and compare visually similar clusters.

In [ ]:
# ── Compute pairwise distances between cluster means ──────────────────

means_flat = cluster_means.reshape(n_clusters, -1)
from scipy.spatial.distance import cdist

distances = cdist(means_flat, means_flat, metric="euclidean")
np.fill_diagonal(distances, np.inf)  # ignore self-similarity

print("Most similar cluster pairs:")
for _ in range(5):
    i, j = np.unravel_index(np.argmin(distances), distances.shape)
    print(f"  Clusters {i} and {j}: distance = {distances[i, j]:.3f}")
    distances[i, j] = np.inf
    distances[j, i] = np.inf

In [ ]:
# ── Side-by-side comparison of similar clusters ───────────────────────

CLUSTER_A = 0  # First cluster to compare
CLUSTER_B = 1  # Second cluster to compare

fig, axes = plt.subplots(2, 6, figsize=(12, 4))

for row, cid in enumerate([CLUSTER_A, CLUSTER_B]):
    mask = cluster_labels == cid
    indices = np.where(mask)[0]
    
    # Mean
    axes[row, 0].imshow(cluster_means[cid], cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_title(f"#{cid} Mean\n({cluster_counts[cid]} chars)", fontsize=9)
    axes[row, 0].set_axis_off()
    
    # Samples
    sample_idx = np.random.choice(indices, size=min(5, len(indices)), replace=False)
    for i, idx in enumerate(sample_idx):
        axes[row, i + 1].imshow(char_imgs[idx], cmap="gray", vmin=0, vmax=1)
        axes[row, i + 1].set_title(char_labels[idx], fontsize=8)
        axes[row, i + 1].set_axis_off()

fig.suptitle(f"Comparing Clusters {CLUSTER_A} vs {CLUSTER_B}", fontweight="bold")
fig.tight_layout()
plt.show()

---
## 5. Document Comparison

Compare clustering results across multiple documents.

In [ ]:
# ── List all documents with clustering results ────────────────────────

charnet_root = Path(CHARNET_ROOT)

docs_with_clusters = []
for doc_path in sorted(charnet_root.iterdir()):
    if doc_path.is_dir() and (doc_path / "clusters_all.npz").exists():
        data = np.load(str(doc_path / "clusters_all.npz"), allow_pickle=True)
        docs_with_clusters.append({
            "name": doc_path.name,
            "n_chars": len(data["cluster_labels"]),
            "n_clusters": len(data["cluster_means"]),
        })

if docs_with_clusters:
    df_docs = pd.DataFrame(docs_with_clusters)
    print(f"Documents with clustering: {len(df_docs)}")
    df_docs
else:
    print("No documents with clustering results found.")

In [ ]:
# ── Compare cluster means across two documents ────────────────────────

DOC_A = DOCUMENT  # First document
DOC_B = None      # Second document (set to a valid name from list above)

if DOC_B is not None and DOC_B != DOC_A:
    data_a = np.load(str(charnet_root / DOC_A / "clusters_all.npz"), allow_pickle=True)
    data_b = np.load(str(charnet_root / DOC_B / "clusters_all.npz"), allow_pickle=True)
    
    means_a = data_a["cluster_means"]
    means_b = data_b["cluster_means"]
    
    # Compute cross-document distances
    flat_a = means_a.reshape(len(means_a), -1)
    flat_b = means_b.reshape(len(means_b), -1)
    cross_dist = cdist(flat_a, flat_b, metric="euclidean")
    
    # Find most similar pairs
    print(f"Most similar clusters between {DOC_A} and {DOC_B}:")
    for _ in range(5):
        i, j = np.unravel_index(np.argmin(cross_dist), cross_dist.shape)
        print(f"  {DOC_A}#{i} <-> {DOC_B}#{j}: distance = {cross_dist[i, j]:.3f}")
        cross_dist[i, j] = np.inf
else:
    print("Set DOC_B to a different document to compare.")